# IPL Data Visualization & Performance Analysis (2008–2020)

**Course:** Data Visualization (2nd Semester)  
**Timeline:** Feb 2024 – Mar 2024  

This notebook loads IPL ball-by-ball data, computes key player/team KPIs, and generates portfolio-ready charts that match the project deliverables (report + presentation).

> **Note:** This is an academic analytics project (not a betting/prediction system).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BALL_PATH = 'IPL Ball-by-Ball 2008-2020_Cleaned.csv'
FINALS_PATH = 'Finals+Awards_Finals.csv'

# Finals/Awards is tab-separated UTF-16
finals = pd.read_csv(FINALS_PATH, encoding='utf-16', sep='	')
finals.columns = [c.strip() for c in finals.columns]

# Ball-by-ball cleaned data
bb = pd.read_csv(BALL_PATH)

bb.head()

## 1) IPL Titles by Team
Counts the final winners from the Finals/Awards table.

In [ ]:
titles = finals['Winners'].value_counts().sort_values(ascending=False)

top_titles = titles.head(10)
plt.figure(figsize=(12,6))
plt.barh(top_titles.index[::-1], top_titles.values[::-1])
plt.title('IPL Titles by Team (Final Winners)')
plt.xlabel('Number of Titles')
plt.ylabel('Team')
plt.tight_layout()
plt.show()

## 2) Top Run Scorers (Ball-by-ball)
Total runs scored by batsmen across all deliveries.

In [ ]:
top_runs = (bb.groupby('batsman')['batsman_runs']
            .sum()
            .sort_values(ascending=False)
            .head(15))

plt.figure(figsize=(12,7))
plt.barh(top_runs.index[::-1], top_runs.values[::-1])
plt.title('Top Run Scorers (2008–2020 Ball-by-ball)')
plt.xlabel('Total Runs')
plt.ylabel('Batsman')
plt.tight_layout()
plt.show()

## 3) Top Wicket Takers
Wickets credited to the bowler (excluding run-outs / retired hurt).

In [ ]:
exclude = {'run out','retired hurt','retired out','obstructing the field'}
wickets = bb[(bb['is_wicket']==1) & (~bb['dismissal_kind'].fillna('').str.lower().isin(exclude))]

top_wkts = (wickets.groupby('bowler')
            .size()
            .sort_values(ascending=False)
            .head(15))

plt.figure(figsize=(12,7))
plt.barh(top_wkts.index[::-1], top_wkts.values[::-1])
plt.title('Top Wicket Takers (Bowler-attributed)')
plt.xlabel('Wickets')
plt.ylabel('Bowler')
plt.tight_layout()
plt.show()

## 4) Strike Rate vs Boundaries (4s + 6s)
A simple relationship view for top batsmen by total runs.

In [ ]:
batsman_summary = bb.groupby('batsman').agg(
    runs=('batsman_runs','sum'),
    balls=('ball','count'),
    fours=('batsman_runs', lambda s: (s==4).sum()),
    sixes=('batsman_runs', lambda s: (s==6).sum())
)

batsman_summary['boundaries'] = batsman_summary['fours'] + batsman_summary['sixes']
batsman_summary['strike_rate'] = (batsman_summary['runs'] / batsman_summary['balls']) * 100

# focus on top batsmen so the plot is readable
plot_df = batsman_summary.sort_values('runs', ascending=False).head(25).copy()

plt.figure(figsize=(10,7))
plt.scatter(plot_df['boundaries'], plot_df['strike_rate'])
plt.title('Strike Rate vs Boundaries (Top 25 Run Scorers)')
plt.xlabel('Boundaries (4s + 6s)')
plt.ylabel('Strike Rate')

# label a few highest run scorers
for name, row in plot_df.head(10).iterrows():
    plt.annotate(name, (row['boundaries'], row['strike_rate']), fontsize=8)

plt.tight_layout()
plt.show()

## Notes / Next Improvements
- Add IPL *matches* dataset to reproduce toss decision → win analysis at match level.
- Publish Tableau dashboard link (or screenshots) for interactive filtering.
- Add season-wise segmentation for player performance trends.